# M06-01 — Explain y DAG

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-cache-particionado.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Separar **tres cosas** que en la UI se parecen:

1. **Plan** — la receta (`explain` la imprime; **no** lee el Parquet).
2. **Job** — una ejecución de verdad (`count`, `show`, `write`).
3. **DAG** — el dibujo de ese job en Spark UI. Solo existe **después** de una acción.

Al terminar: un `print` del DataFrame no mueve Jobs; un `count` sí; y en el `explain` sabes señalar el *Scan* y el *Filter*.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M06-01-explain-dag.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


## Léelo antes de copiar código (si no, el lab no se entiende)

Spark es **vago**: `where` / `select` / `read` **encadenan** trabajo, no lo hacen. Hasta que no lanzas una **acción**, no hay recorrido de ficheros ni número de filas.

| Escribes | Nombre | ¿Lee `fact_lines`? | ¿Sale un número / tabla? |
|----------|--------|--------------------|---------------------------|
| `df.where(...)` | transformación | no | no |
| `print(df)` | enseñar el **objeto** | no | el tipo y las columnas, no el count |
| `df.explain(...)` | imprimir el **plan** | no | un muro de texto (la receta) |
| `df.count()` / `df.show()` | **acción** | sí | un entero / filas |

`explain` **no** es una acción. No llena cache, no crea job, no sube el Job Id. Sirve para leer el mapa **antes** (o después) de viajar.

### Qué significa cada palabro del plan

No memorices Catalyst. En el texto de `explain("formatted")` busca **estas** palabras:

| Si ves… | Quiere decir… |
|---------|----------------|
| *FileScan parquet* / *Scan parquet* | “voy a abrir este directorio Parquet” (ahí sale la ruta `fact_lines`) |
| *PushedFilters* | “estos `where` se los mando al scan” (no leo todo y filtro después del todo) |
| *Filter* | un `where` que aún se aplica como paso |
| *Project* | un `select` / columnas que se quedan |
| *HashAggregate* | un `groupBy` / `count` / `sum` |
| *InMemoryTableScan* | “esto lo leo del **cache**”, no del Parquet (eso es el lab siguiente) |

### Spark UI (útil, no obligatorio)

Pestaña **Ports** del Codespace → puerto **4040** (si está muerto, otra sesión ocupó el puerto: `spark.stop()` y `get_spark` otra vez; a veces cae en 4041).

- **Jobs**: cada `count`/`show` añade una fila. Un `print(df)` o un `explain` **no**.
- **Storage**: este lab no cachea; debe seguir vacío.
- El **DAG** es el grafito que se abre al cliclar un job. Es el dibujo de **esa** acción, no del `explain`.

Si no te abre la UI, da igual: este lab se demuestra con `print` vs `count` vs `explain` en el notebook.

Necesitas `data/staging/fact_lines` (M03-02 o `python3 scripts/run_pipeline.py`).


### Paso 1 — Encadenar no ejecuta

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Celda 0 + sesión + un DataFrame con **tres** `where` y un `select`.

`print(planned)` enseña el objeto (`DataFrame[order_id: …]`). Eso **no** es el número de filas. Spark aún no ha abierto el Parquet (salvo, a veces, para leer el *schema*; eso no cuenta como “ya filtró”).

Si tienes UI: anota el Job Id más alto **antes** de ejecutar, y otra vez **después**. No debe subir.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `DataFrame[...]` con esas cuatro columnas. **Ningún** entero tipo 800. El Job Id de la UI no sube (si la tienes).

**Por qué este paso.** Si `print(planned)` ya te diera 1127, estarías ejecutando un `count` sin darte cuenta.

**Si no sale.** PATH / AnalysisException de `fact_lines` → no está el Parquet de M03. Corre el pipeline o cierra M03-02.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col

spark = get_spark("novashop-m06")

# read + 3 where + select = plan más largo. Todavía NO hay job.
planned = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .where(col("is_billable"))                          # cobrable
    .where(col("gmv_line") > 0)                         # GMV positivo
    .where(col("channel_norm").isin("web", "app"))      # solo esos canales
    .select("order_id", "customer_id", "gmv_line", "order_month")
)

# Objeto, no resultado. Tiene que salir algo tipo DataFrame[order_id: string, ...]
print("¿qué es?", type(planned).__name__)
print(planned)


## Prueba tú — print no cuenta; count sí

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

En *otra* celda, imprime otra vez `planned` y **después** `planned.count()`. Mira qué línea da el entero.

**Qué tienes que ver.** La primera línea sigue siendo el DataFrame. La segunda es un **entero** (varios cientos: cobrable + GMV>0 + web/app). Si tienes UI, **aquí** aparece un job nuevo y un DAG.


In [ ]:
print("otra vez el objeto:", planned)
n = planned.count()
print("ahora sí, filas =", n)


### Paso 2 — Una acción = un job (y entonces hay DAG)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`count()` recorre las particiones, aplica los tres filtros y devuelve un número.

Eso es el viaje: Spark UI → **Jobs** → el job nuevo → clic → **DAG**. El DAG es el dibujo de *este* count, no de los `where` del paso 1.

Si cada celda te crea un job, tienes un `.show()` de debug por medio: coméntalo mientras mides.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Un entero **> 0**, del orden de varios cientos. Un job nuevo en la UI. El DAG tiene al menos un stage.

**Por qué este paso.** Sin acción no hay DAG que mirar. El `explain` del paso 3 es el mapa en texto, no ese dibujo.


In [ ]:
# Acción: ahora sí lee fact_lines y filtra
n = planned.count()
print("líneas cobrables web/app con GMV > 0 =", n)
print("(vuelve a dar el mismo número: el plan no ha cambiado)")


### Paso 3 — `explain` imprime el mapa (no el resultado)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`explain("formatted")` **planifica** (resuelve nombres, optimiza) y **imprime**. No cuenta filas.

En la salida, **no** leas de arriba abajo como un libro. Busca con Ctrl+F:

1. `fact_lines` o `FileScan` / `Scan parquet` → de dónde salen las filas.
2. `is_billable` o `gmv_line` o `channel_norm` o `PushedFilters` / `Filter` → tus `where`.
3. `Project` → el `select` de cuatro columnas.

Copia a Markdown **dos** líneas: una del scan y una del filtro. Con eso basta.

`explain` no debe crear job nuevo.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Texto largo con Scan/FileScan y la ruta `fact_lines`. Los predicados de los `where` aparecen (en *Filter* o en *PushedFilters*). No sale el entero del count.

**Por qué este paso.** Si no ves `is_billable`, estás explain-ando **otro** DataFrame (uno sin ese `where`).


In [ ]:
# Mapa, no viaje. Job Id no debería subir.
planned.explain("formatted")


## Prueba tú — Un where extra no lanza job

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Encadena **otro** `where` **sin** `count`. Imprime el objeto y, si quieres, un `explain`. Luego mira Jobs.

**Qué tienes que ver.** Otro `DataFrame[...]`. Cero jobs nuevos. El count de `mas_estrecho` (si lo lanzas) será **menor** que el de `planned`; eso ya sería una acción nueva.


In [ ]:
# Transformación de más: el plan crece, el job no
mas_estrecho = planned.where(col("gmv_line") > 50)
print("sigue siendo un objeto:", mas_estrecho)
# mas_estrecho.explain("formatted")  # opcional: un Filter/PushedFilters más
print("si tienes UI: el Job Id más alto no cambia respecto al count del paso 2")


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Markdown en *tu* notebook con:

- una frase: `print(planned)` no lee datos; `count()` sí; `explain` imprime el plan.
- las **dos** líneas del explain (scan + filtro).
- si usaste UI: Job Id antes del primer count vs después (solo sube con el count).


## Mejora — formatted vs extended

`explain("formatted")` es el físico, legible. `explain(True)` (o `explain("extended")`) suelta **cuatro** capas: parsed → analyzed → optimized → physical. Más ruido.

Ejecuta los dos. En el físico / formatted, ¿ves `PushedFilters` junto al FileScan? Eso es “el `where` se empujó al scan”. Anótalo.

Si el filtro **no** aparece, lo pusiste *después* de un `select` que ya tiró la columna.

Si te atasca, el código está en la celda siguiente.


In [ ]:
print("===== formatted (el de clase) =====")
planned.explain("formatted")
print("===== extended (cuatro capas; busca Physical Plan y PushedFilters) =====")
planned.explain(True)


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| UI vacía / 404 | Puerto 4040 no reenviado o sesión en 4041 | Ports → 4040; o `spark.stop()` + `get_spark()` |
| Cada celda crea un job | Tienes un `show`/`count` de debug | Coméntalo mientras mides |
| Dos sesiones | `SparkSession()` a mano | Solo `get_spark()` |
| explain peta AnalysisException | Columna mal escrita | El plan también resuelve nombres: corrige el typo |
| No está fact_lines | Saltaste M03 | `run_pipeline.py` o lab M03-02 |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M06-02 cache](03-lab-cache-particionado.ipynb).
